In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error

RANDOM_STATE = 42
TARGET = "ClosePrice"

In [75]:
train_df = pd.read_parquet("../data/train_preprocessed.parquet")
test_df = pd.read_parquet("../data/test_preprocessed.parquet")
df = pd.read_parquet("../data/full_data_preprocessed.parquet")
print(f"Train dataset: {train_df.shape}, Test dataset: {test_df.shape}, Full Dataset: {df.shape} \n")

districts_raw = gpd.read_file("../data/California_School_District_Areas_2024-25.geojson")
print(f"Loaded {len(districts_raw)} districts from California School District Areas 2024-2025")
print(f"District Type Unique Entries: {districts_raw["DistrictType"].unique()}")

Train dataset: (81857, 30), Test dataset: (7442, 30), Full Dataset: (240901, 30) 

Loaded 937 districts from California School District Areas 2024-2025
District Type Unique Entries: ['Unified' 'Elementary' 'High']


In [66]:
districts = districts_raw[districts_raw["DistrictType"].isin(["Unified", "High"])].copy()
districts = districts[["DistrictType", "DistrictName", "geometry"]].to_crs("EPSG:4326")
print(f"Filtered to {len(districts)} Unified/High polygons")

Filtered to 421 Unified/High polygons


In [67]:
# Enhance Geographic layer using CA School District data and property coordinates
def add_spatial_district(frame, districts):
    has_coords = frame["Latitude"].notna() & frame["Longitude"].notna()
    points_gdf = gpd.GeoDataFrame(
        frame.loc[has_coords, []],
        geometry=gpd.points_from_xy(frame.loc[has_coords, "Longitude"], frame.loc[has_coords, "Latitude"]),
        crs="EPSG:4326",
    )
    joined = gpd.sjoin(points_gdf, districts, how="left", predicate="within")
    joined = joined[~joined.index.duplicated(keep="first")]  # defensive: boundary edge overlaps

    spatial_col = pd.Series(np.nan, index=frame.index, dtype=object)
    spatial_col.loc[joined.index] = joined["DistrictName"].values
    return spatial_col.fillna(frame["HighSchoolDistrict"])  # fall back where no polygon matched

train_df["SchoolDistrictSpatial"] = add_spatial_district(train_df, districts)
test_df["SchoolDistrictSpatial"] = add_spatial_district(test_df, districts)

print(f"Train coverage: {train_df['SchoolDistrictSpatial'].notna().mean():.1%}")
print(f"Test coverage:  {test_df['SchoolDistrictSpatial'].notna().mean():.1%}")

Train coverage: 100.0%
Test coverage:  100.0%


In [68]:
raw_cols_needed = ["ListingKey", "BedroomsTotal", "BathroomsTotalInteger", "YearBuilt"]
raw_lookup = df[raw_cols_needed]
raw_lookup = raw_lookup.rename(columns={
    "BedroomsTotal": "BedroomsTotal_raw",
    "BathroomsTotalInteger": "BathroomsTotalInteger_raw",
    "YearBuilt": "YearBuilt_raw",
})

In [69]:
train_df = train_df.merge(raw_lookup, on="ListingKey", how="left")
test_df = test_df.merge(raw_lookup, on="ListingKey", how="left")

print("Unmatched in train:", train_df["BedroomsTotal_raw"].isna().sum())
print("Unmatched in test:", test_df["BedroomsTotal_raw"].isna().sum())

Unmatched in train: 0
Unmatched in test: 0


In [70]:
# Engineer Bed-Bath Ratio Feature
train_df["BedBathRatio"] = train_df["BedroomsTotal_raw"] / train_df["BathroomsTotalInteger_raw"].replace(0, np.nan)
test_df["BedBathRatio"] = test_df["BedroomsTotal_raw"] / test_df["BathroomsTotalInteger_raw"].replace(0, np.nan)

# Engineer Propert Age Feature
train_df["PropertyAgeAtSale"] = (train_df["CloseDate"].dt.year - train_df["YearBuilt_raw"]).clip(lower=0)
test_df["PropertyAgeAtSale"] = (test_df["CloseDate"].dt.year - test_df["YearBuilt_raw"]).clip(lower=0)

train_medians = {"BedBathRatio": train_df["BedBathRatio"].median(), "PropertyAgeAtSale": train_df["PropertyAgeAtSale"].median()}
for col, fallback in train_medians.items():
    train_df[col] = train_df[col].fillna(fallback)
    test_df[col] = test_df[col].fillna(fallback)  # train stat applied to both, same convention as elsewhere

print(train_df[["BedBathRatio", "PropertyAgeAtSale"]].describe())

       BedBathRatio  PropertyAgeAtSale
count  82030.000000       82030.000000
mean       1.445512          44.796855
std        0.444270          26.039067
min        0.000000           0.000000
25%        1.000000          24.000000
50%        1.500000          44.000000
75%        1.500000          66.000000
max        6.000000         249.000000


In [71]:
NON_FEATURE_COLS = ["ListingKey", "CloseDate", "BedroomsTotal_raw", "BathroomsTotalInteger_raw", "YearBuilt_raw"]
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "HighSchoolDistrict", "SchoolDistrictSpatial", "Levels"]
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns]

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
encoder.fit(train_df[CATEGORICAL_COLS])

def build_X_y(frame):
    numeric_part = frame.drop(columns=[c for c in [TARGET] + NON_FEATURE_COLS + CATEGORICAL_COLS if c in frame.columns])
    numeric_part = numeric_part.select_dtypes(include=[np.number])
    encoded = encoder.transform(frame[CATEGORICAL_COLS])
    X = hstack([csr_matrix(numeric_part.values), encoded]).tocsr()
    y = frame[TARGET].values
    feature_names = list(numeric_part.columns) + list(encoder.get_feature_names_out(CATEGORICAL_COLS))
    return X, y, feature_names

X_train, y_train, feature_names = build_X_y(train_df)
X_test, y_test, _ = build_X_y(test_df)
print(f"{X_train.shape[1]} features")

3830 features


In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, name):
    train_pred, test_pred = model.predict(X_train), model.predict(X_test)
    return {"model": name, "train_r2": r2_score(y_train, train_pred), "test_r2": r2_score(y_test, test_pred),
            "test_mae": mean_absolute_error(y_test, test_pred), "test_rmse": root_mean_squared_error(y_test, test_pred),
            "test_mape": mean_absolute_percentage_error(y_test, test_pred)}

new_results = []
linear_model = LinearRegression().fit(X_train, y_train)
new_results.append(evaluate_model(linear_model, X_train, y_train, X_test, y_test, "Linear Regression (baseline)"))

tree_unconstrained = DecisionTreeRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
new_results.append(evaluate_model(tree_unconstrained, X_train, y_train, X_test, y_test, "Decision Tree (unconstrained)"))

tree_tuned = DecisionTreeRegressor(max_depth=12, min_samples_leaf=20, random_state=RANDOM_STATE).fit(X_train, y_train)
new_results.append(evaluate_model(tree_tuned, X_train, y_train, X_test, y_test, "Decision Tree (max_depth=12, min_leaf=20)"))

rf_model = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE).fit(X_train, y_train)
new_results.append(evaluate_model(rf_model, X_train, y_train, X_test, y_test, "Random Forest"))

new_results_df = pd.DataFrame(new_results).round(4)
new_results_df["feature_set"] = "New (+ school district, bed-bath ratio, age)"
new_results_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,feature_set
0,Linear Regression (baseline),0.8577,0.8467,183650.3396,308501.8730,0.1822,"New (+ school district, bed/bath ratio, age)"
1,Decision Tree (unconstrained),0.9999,0.7752,187768.9641,373625.6114,0.1545,"New (+ school district, bed/bath ratio, age)"
2,"Decision Tree (max_depth=12, min_leaf=20)",0.8421,0.7969,194659.1944,355084.6571,0.1661,"New (+ school district, bed/bath ratio, age)"
3,Random Forest,0.9378,0.8737,143306.3777,280009.3747,0.1188,"New (+ school district, bed/bath ratio, age)"


Adding the School District Layer, Bed-Bath Ratio, and Age features increased all train and test metrics (ex: Lin Reg test_r2 went from 0.8420 -> 0.8467, while DT test_r2 went from 0.7961 -> 0.7969). It is clear the increase in the linear regression metrics were far more significant than the decision tree and random forest metric increases, as expected with comprehensive tree-based models vs. linear regression.

In [76]:
coord_lookup = df[["ListingKey", "Latitude", "Longitude"]] # Lat and Long are already normalized, so we go back to the unscaled or encded data, df
coord_lookup = coord_lookup.rename(columns={"Latitude": "Latitude_raw", "Longitude": "Longitude_raw"})

train_df = train_df.merge(coord_lookup, on="ListingKey", how="left")
test_df = test_df.merge(coord_lookup, on="ListingKey", how="left")

COMP_K = 15     # nearest 15 neighbors by long and lat
coord_cols = ["Latitude_raw", "Longitude_raw"]
train_coords = train_df[coord_cols].values
train_prices = train_df["ClosePrice"].values

nn = NearestNeighbors(n_neighbors=COMP_K + 1, algorithm="ball_tree").fit(train_coords)
_, neighbor_idx = nn.kneighbors(train_coords)
neighbor_idx = neighbor_idx[:, 1:]  # drop self
train_df["CompPriceKNN"] = train_prices[neighbor_idx].mean(axis=1)

nn_test = NearestNeighbors(n_neighbors=COMP_K, algorithm="ball_tree").fit(train_coords)
_, test_neighbor_idx = nn_test.kneighbors(test_df[coord_cols].values)
test_df["CompPriceKNN"] = train_prices[test_neighbor_idx].mean(axis=1)

print(train_df["CompPriceKNN"].describe())

count    8.203000e+04
mean     1.042003e+06
std      6.459421e+05
min      2.548000e+05
25%      5.996003e+05
50%      8.503267e+05
75%      1.262832e+06
max      5.103433e+06
Name: CompPriceKNN, dtype: float64


In [78]:
# Idea from Fourier Transformations:
# lets the model recognize December and January as adjacent rather than maximally distant.

close_month_period = pd.PeriodIndex(train_df["CloseMonth"], freq="M")
test_month_period = pd.PeriodIndex(test_df["CloseMonth"], freq="M")

all_months_sorted = sorted(set(close_month_period) | set(test_month_period))
month_to_trend = {m: i for i, m in enumerate(all_months_sorted)}

train_df["MonthsSinceStart"] = close_month_period.map(month_to_trend)
test_df["MonthsSinceStart"] = test_month_period.map(month_to_trend)

train_df["MonthSin"] = np.sin(2 * np.pi * close_month_period.month / 12)
train_df["MonthCos"] = np.cos(2 * np.pi * close_month_period.month / 12)
test_df["MonthSin"] = np.sin(2 * np.pi * test_month_period.month / 12)
test_df["MonthCos"] = np.cos(2 * np.pi * test_month_period.month / 12)

In [81]:
NON_FEATURE_COLS = ["ListingKey", "CloseDate", "BedroomsTotal_raw", "BathroomsTotalInteger_raw", "YearBuilt_raw"]
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "HighSchoolDistrict", "SchoolDistrictSpatial", "Levels"]
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns]

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
encoder.fit(train_df[CATEGORICAL_COLS])

def build_X_y(frame):
    numeric_part = frame.drop(columns=[c for c in [TARGET] + NON_FEATURE_COLS + CATEGORICAL_COLS if c in frame.columns])
    numeric_part = numeric_part.select_dtypes(include=[np.number])
    encoded = encoder.transform(frame[CATEGORICAL_COLS])
    X = hstack([csr_matrix(numeric_part.values), encoded]).tocsr()
    y = frame[TARGET].values
    feature_names = list(numeric_part.columns) + list(encoder.get_feature_names_out(CATEGORICAL_COLS))
    return X, y, feature_names

X_train, y_train, feature_names = build_X_y(train_df)
X_test, y_test, _ = build_X_y(test_df)
print(f"{X_train.shape[1]} features")

3429 features


In [82]:
def evaluate_model(model, X_train, y_train, X_test, y_test, name):
    train_pred, test_pred = model.predict(X_train), model.predict(X_test)
    return {"model": name, "train_r2": r2_score(y_train, train_pred), "test_r2": r2_score(y_test, test_pred),
            "test_mae": mean_absolute_error(y_test, test_pred), "test_rmse": root_mean_squared_error(y_test, test_pred),
            "test_mape": mean_absolute_percentage_error(y_test, test_pred)}

new_results = []
linear_model = LinearRegression().fit(X_train, y_train)
new_results.append(evaluate_model(linear_model, X_train, y_train, X_test, y_test, "Linear Regression (baseline)"))

tree_unconstrained = DecisionTreeRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
new_results.append(evaluate_model(tree_unconstrained, X_train, y_train, X_test, y_test, "Decision Tree (unconstrained)"))

tree_tuned = DecisionTreeRegressor(max_depth=12, min_samples_leaf=20, random_state=RANDOM_STATE).fit(X_train, y_train)
new_results.append(evaluate_model(tree_tuned, X_train, y_train, X_test, y_test, "Decision Tree (max_depth=12, min_leaf=20)"))

rf_model = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE).fit(X_train, y_train)
new_results.append(evaluate_model(rf_model, X_train, y_train, X_test, y_test, "Random Forest"))

new_results_df = pd.DataFrame(new_results).round(4)
new_results_df["feature_set"] = "New (+ school district, bed-bath ratio, age)"
new_results_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,feature_set
0,Linear Regression (baseline),0.8069,0.8097,208494.3137,343761.6985,0.1967,"New (+ school district, bed-bath ratio, age)"
1,Decision Tree (unconstrained),1.0000,0.7963,187334.9455,355641.1035,0.1522,"New (+ school district, bed-bath ratio, age)"
2,"Decision Tree (max_depth=12, min_leaf=20)",0.8925,0.8515,163278.4799,303635.0329,0.1347,"New (+ school district, bed-bath ratio, age)"
3,Random Forest,0.9480,0.8842,138834.2571,268125.3582,0.1140,"New (+ school district, bed-bath ratio, age)"


While linear regression performance decreases with the additional CompPriceKNN variable and Cyclical Month Encoding, the DT and RF models benefit greatly. 

Example:
- Before Feature Engineering: RF test_r2 = `0.8734`
- After Feature Engineering: RF test_r2 = `0.8842`